# Near-to-far planning-cost adapter pilot

Run on a **Colab GPU**. This uses one frozen matched-v3 fixed world model and one image goal. The first run is 120 episodes: 40 raw evaluations, 40 fresh near attempts, 40 adapted evaluations. Two extra seeds run only if the first screen passes. No historical latent cache is needed.

Copy the delivered `curriculum-cost-adapter-source.zip` to `MyDrive/datasets/curriculum_cost_adapter/source/`. The source bundle includes its original Git revision. It works before the local branch is published. All experiment outputs go to a new Drive directory; model copies and source live under `/content`.

The notebook stops adaptation if raw near-goal success is below 12/20. That is a result to inspect, not a reason to select easier test cases. See `docs/curriculum_cost_adapter.md` in the bundle for the exact protocol and training defaults.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, shutil, signal, subprocess, sys, zipfile
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
DRIVE = Path('/content/drive/MyDrive')
SOURCE_ZIP = DRIVE / 'datasets/curriculum_cost_adapter/source/curriculum-cost-adapter-source.zip'
REPO = Path('/content/curriculum-cost-adapter')
DINO_ROOT = Path('/content/dino_wm')
OUTPUT = DRIVE / 'datasets/curriculum_cost_adapter/pilot_v1'
ENCODER_DRIVE = DRIVE / 'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
WM_DRIVE = DRIVE / 'datasets/wall_matched_fixed_multi_v3/fixed/world_model/best.pt'
ARTIFACTS = Path('/content/curriculum-cost-adapter-artifacts')
DINO_COMMIT = '0a9492fa12044b852ae9e001cc74604b79c8bb0c'

def shell(command, *, cwd=None, log=None):
    command = [str(arg) for arg in command]
    print('$', ' '.join(command), flush=True)
    stream = open(log, 'a', buffering=1) if log else None
    try:
        with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    if stream:
                        stream.write(line)
                returncode = process.wait()
            except BaseException:
                process.send_signal(signal.SIGINT)
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.terminate()
                raise
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    finally:
        if stream:
            stream.close()


## Load the delivered source and simulator

Rerunning this cell reuses the same bundle. It refuses to mix an existing source directory with a different bundle. For a different source revision or protocol, choose a fresh experiment output directory.


In [ ]:
assert SOURCE_ZIP.is_file(), f'Copy the delivered source ZIP to {SOURCE_ZIP}'
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    prefix = REPO.name + '/'
    for name in archive.namelist():
        assert name.startswith(prefix) and '..' not in Path(name).parts, 'Unexpected archive member'
    bundled = json.loads(archive.read(prefix + 'pilot_source.json'))
    if REPO.exists():
        existing = json.loads((REPO / 'pilot_source.json').read_text())
        assert existing == bundled, 'Existing /content source belongs to another bundle'
    else:
        archive.extractall('/content')
print('Pilot source revision:', bundled['commit'])

shell([sys.executable, '-m', 'pip', 'install', '-q', 'gym==0.26.2', 'omegaconf',
       'pyyaml', 'scipy', 'einops', 'matplotlib', 'imageio', 'imageio-ffmpeg', 'pillow'])
if not DINO_ROOT.exists():
    shell(['git', 'clone', 'https://github.com/gaoyuezhou/dino_wm.git', DINO_ROOT])
assert not subprocess.check_output(['git', '-C', str(DINO_ROOT), 'status', '--porcelain', '--untracked-files=no'], text=True).strip(), 'DINO-WM source has local changes'
shell(['git', '-C', DINO_ROOT, 'checkout', DINO_COMMIT])

ARTIFACTS.mkdir(parents=True, exist_ok=True)
ENCODER = ARTIFACTS / 'encoder.pth.tar'
CHECKPOINT = ARTIFACTS / 'fixed-best.pt'
for source, destination in ((ENCODER_DRIVE, ENCODER), (WM_DRIVE, CHECKPOINT)):
    assert source.is_file(), f'Missing selected input artifact: {source}'
    if not destination.exists() or destination.stat().st_size != source.stat().st_size:
        shutil.copy2(source, destination)
OUTPUT.mkdir(parents=True, exist_ok=True)
COMMON = ['--output-dir', str(OUTPUT), '--world-model-checkpoint', str(CHECKPOINT),
          '--encoder-checkpoint', str(ENCODER), '--dino-wm-root', str(DINO_ROOT),
          '--device', 'cuda', '--precision', 'fp32', '--encoder-batch-size', '8']

def stage(name, seed=0, extra=()):
    log = OUTPUT / f'{name}_run_{seed}.log'
    shell([sys.executable, '-m', 'src.mylonas_ijepa_wm.curriculum_cost_pilot', name,
           *COMMON, '--run-seed', str(seed), *extra], cwd=REPO, log=log)

print('GPU:', torch.cuda.get_device_name())
print('Torch:', torch.__version__)
print('Outputs:', OUTPUT)


## 1. Freeze the cases and measure identity/raw cost

`prepare` is idempotent for identical settings. There are 20 near and 20 farther evaluation starts, shared across conditions and run seeds. Collection cases are separate. Model hashes, source hashes, precision and version are checked when execution starts.

**If baseline exits with code 2, inspect `gate.json`. Run the diagnostic cell below manually if competence failed, then stop.** Do not continue to collection after a failed gate.


In [ ]:
stage('prepare')
stage('baseline')
gate = json.loads((OUTPUT / 'gate.json').read_text())
print(gate)
assert gate['passed'], 'Near-goal competence gate failed; inspect position diagnostic.'


### Position-cost diagnostic if the gate failed

This uses the model's predicted coordinates. It helps diagnose a poor visual cost versus a broader planning problem; it is not a simulator-dynamics oracle. The additional evaluations are counted separately.


In [ ]:
gate = json.loads((OUTPUT / 'gate.json').read_text())
if not gate['passed']:
    stage('position')
    print('Stop here and inspect the raw and position results before changing the protocol.')
else:
    print('Competence gate passed; position diagnostic skipped.')


## 2. Collect near-goal successes and fit the adapter

Forty identity-cost attempts, including failures, form the collection budget. The adapter is fitted afterward using successful real observations only. It never trains on baseline/evaluation observations. Training uses a fixed 500-update schedule and the final iterate; farther evaluation cannot select a checkpoint.

Every primitive action is journaled so interrupted attempts remain in the interaction budget. Collection and completed episode evaluation resume automatically. Interrupted fitting restarts its fixed schedule from identity using the saved replay.


In [ ]:
assert json.loads((OUTPUT / 'gate.json').read_text())['passed']
stage('collect', seed=0)
stage('train', seed=0)


## 3. Test farther transfer and near retention

The learned adapter is frozen during evaluation. The raw and adapted planners use identical cases, CEM seeds and search settings. Success counts and paired wins/losses matter; lower training loss or better real-frame correlation alone is insufficient.


In [ ]:
stage('evaluate', seed=0)
stage('report')
report = json.loads((OUTPUT / 'report.json').read_text())
print(json.dumps(report['runs']['0'], indent=2))
from IPython.display import display, Image
display(Image(filename=str(OUTPUT / 'success_by_band.png')))
display(Image(filename=str(OUTPUT / 'run_0/trajectories.png')))
display(Image(filename=str(OUTPUT / 'run_0/training_loss.png')))


## Optional diagnostic: real maps and CEM candidate rankings

This compares predicted and real costs for initial samples, final elites, and the returned elite mean. Terminal replays deliberately continue after success to match the predicted horizon. Default audits add **1,700 primitive interactions**, kept separate from adaptation. They use one near and one farther case under each cost.


In [ ]:
RUN_DIAGNOSTICS = True
if RUN_DIAGNOSTICS:
    stage('diagnostics', seed=0)
    display(Image(filename=str(OUTPUT / 'diagnostics/run_0/real_cost_map.png')))
    print((OUTPUT / 'diagnostics/run_0/summary.json').read_text())


## Conditional replication

The first-run engineering screen is at least four extra farther successes with no more than two lost near successes. It motivates replication, not a significance claim. Runs 1 and 2 collect their own 40 near attempts and initialize/train independently. Evaluation reuses the same 20 cases per band, so these are not 60 independent tasks. The three-run experiment totals 280 episodes before optional diagnostics.


In [ ]:
report = json.loads((OUTPUT / 'report.json').read_text())
if report['runs']['0']['screen_passed']:
    for seed in (1, 2):
        stage('collect', seed=seed)
        stage('train', seed=seed)
        stage('evaluate', seed=seed)
    stage('report')
    display(Image(filename=str(OUTPUT / 'success_by_band.png')))
else:
    print('First-run screen did not pass. Inspect failures and diagnostics before expanding the experiment.')


## Optional videos from saved collection frames

This creates one collection success video and one failure video when available. It does not execute more actions. For evaluation videos, pass `--videos` during the original evaluation stage; rerunning a completed stage will reuse its records.


In [ ]:
sys.path.insert(0, str(REPO))
from src.mylonas_ijepa_wm.plan_pointmaze import save_episode_video
from IPython.display import Video
records = [json.loads(path.read_text()) for path in sorted((OUTPUT / 'episodes/run_0/collect').glob('*.json'))]
for desired in (True, False):
    selected = next((row for row in records if row['success'] == desired), None)
    if selected:
        raw = torch.load(OUTPUT / selected['raw_path'], map_location='cpu', weights_only=True)
        path = save_episode_video([frame.numpy() for frame in raw['frames']], OUTPUT / 'videos/collection_examples',
                                  int(desired), selected['case']['seed'], desired, 10)
        display(Video(str(path), embed=True))
